In [1]:
# import libraries
import pandas as pd
import schedule
import os, re
from datetime import datetime
import time
from time_diff_function import timediff

In [2]:
pthBESAFiles = r'P:\Shared Folders\Data\MTM'                                                           # BESA files
pthMDO       = r'\\Pim-cpt-statpro\Plugins\Profiles\PIM\DataStage\Data\manual\MarketDataOverrides.xls' # eagle mod durns

In [3]:
# get data in MarketDataOverrides.xls file
mdo          = pd.read_excel(pthMDO)                         # eagle MarketDataOverrides.xls file

# get latest BESA data file
besa_fnames  = [int(s[re.search('\d{8}',s).span()[0]:re.search('\d{8}',s).span()[0]+8]) for s in \
                os.listdir(pthBESAFiles) if "." in s]
besa_fdate   = max(besa_fnames)                              # BESA file date as an integer
bsaDate      = datetime.strptime(str(besa_fdate), "%Y%m%d")  # BESA file date as a datetime
res          = list(filter(lambda x: str(besa_fdate) in x, os.listdir(pthBESAFiles)))[0]
bsa          = os.path.join(pthBESAFiles, res)
besa_data    = pd.read_csv(  bsa,                                skiprows   =  4,   \
                           usecols = ['Bond Code', 'Modified Duration']).dropna() # BESA-listed security codes

print(f' Latest BESA file date : {bsaDate.strftime("%a %d %b %Y")}, {len(besa_data)} entries')

 Latest BESA file date : Tue 16 Apr 2024, 295 entries


In [4]:
# add three new columns filled with 1s to the BESA data
col_names = ['c', 'b', 'a']
for col in col_names:
    besa_data.insert(1, col, 1)

In [5]:
# take on the MarketDataOverrides.csv file column names
besa_data.columns = mdo.columns.tolist()

In [6]:
# get the marker row number
k = mdo[mdo['Instrument Code'] == 'TEST_BK_EXPIRY_1004'].index[0]

# delete rows beneath the marker
mdo.drop(mdo.index[k + 1:], inplace = True)

# concatenate the MArketDataOverides data with the new BESA modified duration data
mdo_new = pd.concat([mdo, besa_data], ignore_index = True, axis = 0)

# drop duplicate instruments https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html
mdo_new.drop_duplicates(subset = ['Instrument Code'], inplace = True, keep = 'last')

In [12]:
# save the new MarketDataOverrides.xls file
mdo_new.to_excel(pthMDO + 'x' , sheet_name = 'MarketDataOverrides', index  = False)

import win32com.client as win32 # library to convert xls to xlsx

# start excel
excel = win32.gencache.EnsureDispatch('Excel.Application')

wb = excel.Workbooks.Open(pthMDO + 'x')
excel.DisplayAlerts = False # suppress Excel warning dialogue
wb.SaveAs(pthMDO, FileFormat = 56) #FileFormat = 56(51) is for .xls(x)
wb.Close()
excel.DisplayAlerts = True  # unsuppress Excel warning dialogue
os.remove(pthMDO + 'x') # delete MarketDataOverrides.xlsx

In [ ]:
# \\Pim-cpt-statpro\Plugins\Profiles\PIM\DataStage\Data\manual\MarketDataOverrides.xls